In [ ]:
%run "/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/code/Initialization/Initialization.ipynb"

In [ ]:
df_rates = pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/TreasuryRates.pkl')

In [ ]:
df_rates.index.rename("Date",inplace=True)

In [ ]:
df_rates.drop(columns={"date"})

In [ ]:
df = pd.merge(df,df_rates,left_index=True, right_index=True)

In [ ]:
df.columns.values

In [ ]:
from scipy.stats import norm
from scipy.optimize import minimize
class BlackScholes:
    """ 
    Class to calculate (European) call and put option prices through the Black-Scholes formula 
    without dividends
    
    :param S: Price of underlying stock
    :param K: Strike price
    :param T: Time till expiration (in years)
    :param r: Risk-free interest rate (0.05 indicates 5%)
    :param sigma: Volatility (standard deviation) of stock (0.15 indicates 15%)
    """
    @staticmethod
    def _d1(S, K, T, r, sigma):
        return (1 / (sigma * np.sqrt(T))) * (np.log(S/K) + (r + sigma**2 / 2) * T)
    
    def _d2(self, S, K, T, r, sigma):
        return self._d1(S, K, T, r, sigma) - sigma * np.sqrt(T)
    
    def call_price(self, S, K, T, r, sigma):
        """ Main method for calculating price of a call option """
        d1 = self._d1(S, K, T, r, sigma)
        d2 = self._d2(S, K, T, r, sigma)
        return norm.cdf(d1) * S - norm.cdf(d2) * K * np.exp(-r*T)
    
    def put_price(self, S, K, T, r, sigma):
        """ Main method for calculating price of a put option """
        d1 = self._d1(S, K, T, r, sigma)
        d2 = self._d2(S, K, T, r, sigma)
        return norm.cdf(-d2) * K * np.exp(-r*T) - norm.cdf(-d1) * S
    
    def call_in_the_money(self, S, K, T, r, sigma):
        """ 
        Calculate probability that call option will be in the money at
        maturity according to Black-Scholes.
        """
        d2 = self._d2(S, K, T, r, sigma)
        return norm.cdf(d2)
    
    def put_in_the_money(self, S, K, T, r, sigma):
        """ 
        Calculate probability that put option will be in the money at
        maturity according to Black-Scholes.
        """
        d2 = self._d2(S, K, T, r, sigma)
        return 1 - norm.cdf(d2)

In [ ]:
df

In [ ]:
def payoff_function(S: float, K: float, T: float, r: float, sigma: float, 
                    alpha: int = 3, beta: int = 1):
    """ Some crazy custom payoff function """
    call_prices = BlackScholes().call_price(S, K, T, r, sigma)
    put_prices = BlackScholes().put_price(S, K, T, r, sigma) 
    return alpha * call_prices**(alpha*put_prices) / (beta + put_prices)**(beta*call_prices)

def payoff_function_float(end_prices: np.ndarray, K: int, T: float, r: float, sigma: float, 
                    alpha: int = 3, beta: int = 1):
    """ Some crazy custom payoff function """
    call_prices = np.array([BlackScholes().call_price(s, K, T, r, sigma) for s in end_prices])
    put_prices = np.array([BlackScholes().put_price(s, K, T, r, sigma) for s in end_prices])
    return alpha * call_prices**(alpha*put_prices) / (beta + put_prices)**(beta*call_prices)

df["2022-06-01"]
K = 35 # Strike price
T = 1 # Time till expiration (in years)
r = 0.05 # Risk-free interest rate
end_prices = np.arange(30, 50, 0.1)
payoffs = payoff_function(end_prices, S, T, r, sigma)

In [ ]:
call_prices=[]
put_prices=[]
for date, row in df.T.iteritems():
    S = row["S&P500"]
    K = row["S&P500"] # Strike price
    T = (1/12) # Time till expiration (in years)
    r = row["year1"] # Risk-free interest rate
    end_prices =row["S&P500"]# np.arange(row["S&P500"]-1, row["S&P500"]+1, 0.1)
    sigma = row["VIX"]
    call_prices.append(BlackScholes().call_price(S, K, T, r, sigma))
    put_prices.append(BlackScholes().put_price(S, K, T, r, sigma))

In [ ]:
df_calls = pd.DataFrame(data=call_prices)
df_puts = pd.DataFrame(data=put_prices)

In [ ]:
df_calls.plot()

In [ ]:
df_puts.plot()